# Taller de Semana 4: exploración, limpieza, estadística y visualización

En este taller aplicarás los conocimientos trabajados en la semana 4 sobre lectura y exploración de datasets, limpieza y preparación de datos, estadística descriptiva y visualización. Trabajarás con una muestra local del dataset público **Resultados Únicos Saber 11** del portal Datos Abiertos Colombia.

**Fuente original del dataset:** `https://www.datos.gov.co/resource/kgxf-xxbe.json`  
**Copia local usada en este taller:** muestra de 2500 registros descargada el **8 de junio de 2026**.

## Habilidades en práctica

Al desarrollar este taller podrás verificar tu progreso para:

**1.** Leer y explorar un dataset real con `pandas`. <br>
**2.** Detectar faltantes, duplicados e inconsistencias de formato. <br>
**3.** Preparar una base de datos para análisis estadístico. <br>
**4.** Interpretar media, mediana, moda, varianza, desviación estándar, cuartiles y percentiles. <br>
**5.** Analizar distribuciones y construir visualizaciones útiles para tomar decisiones.

## Instrucciones

En cada uno de los siguientes ejercicios deberás escribir el código solicitado estrictamente en las celdas indicadas para ello, teniendo en cuenta las siguientes recomendaciones:

* No crear, eliminar o modificar celdas de este Notebook, pues puede verse afectado el proceso de validación.
* Cuando el enunciado pida una función, usa exactamente el nombre indicado.
* Lee con cuidado qué debe retornar cada ejercicio: `DataFrame`, `Series`, entero o gráfico.
* Si una validación falla, vuelve al enunciado y revisa si tu salida tiene el formato correcto.

Este taller está diseñado para que el entorno ya quede listo y tú te concentres en razonar y escribir el código de cada paso.

## Ejercicios

En la siguiente celda se importan los paquetes necesarios, se carga la base de trabajo y se definen algunas ayudas de apoyo para la limpieza.

In [ ]:
# Esta celda no es modificable
import unicodedata
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import numpy.testing as npt
import pandas as pd
from matplotlib.axes import Axes

RUTA_DATOS = Path("Archivos") / "saber11_2500.csv"
saber11 = pd.read_csv(RUTA_DATOS)

def normalizar_texto(valor):
    if pd.isna(valor):
        return valor
    valor = str(valor).strip().upper()
    valor = unicodedata.normalize("NFKD", valor).encode("ascii", "ignore").decode("ascii")
    return valor

saber11.head()

Antes de empezar, observa la estructura del archivo local que se usará a lo largo del taller.

In [ ]:
# Esta celda no es modificable
print("Dimensiones del dataset base:", saber11.shape)
print("\nColumnas disponibles:")
print(list(saber11.columns))

### Ejercicio 1

Implementa una función llamada `cargar_datos` que no reciba parámetros, lea el archivo `"Archivos/saber11_2500.csv"` y retorne un `DataFrame`.

La función debe retornar un `DataFrame`.

In [1]:
# Escribe tu respuesta en esta celda
import pandas as pd

def cargar_datos():
    df = pd.read_csv("Archivos/saber11_2500.csv")
    return df


In [ ]:
## AUTO-REVISION

columnas_esperadas = [
    'periodo', 'estu_consecutivo', 'cole_area_ubicacion', 'cole_bilingue',
    'cole_calendario', 'cole_genero', 'cole_jornada', 'cole_naturaleza',
    'cole_depto_ubicacion', 'cole_mcpio_ubicacion', 'estu_genero',
    'fami_estratovivienda', 'fami_personashogar', 'fami_tienecomputador',
    'fami_tieneinternet', 'desemp_ingles', 'punt_ingles', 'punt_matematicas',
    'punt_sociales_ciudadanas', 'punt_c_naturales', 'punt_lectura_critica',
    'punt_global'
]

try:
    cargar_datos
    assert type(cargar_datos) == type(lambda: None)
except:
    raise NotImplementedError("No existe una función llamada cargar_datos.")

try:
    respuesta = cargar_datos()
except:
    raise RuntimeError("Tu función produce un error al ejecutarse.")

assert isinstance(respuesta, pd.DataFrame), "La función debe retornar un DataFrame."
assert list(respuesta.columns) == columnas_esperadas, "Las columnas del DataFrame no coinciden con las esperadas."
assert respuesta.shape == (2500, 22), "La forma del DataFrame cargado es incorrecta."

print("Auto-revision superada.")

### Ejercicio 2

Implementa una función llamada `resumen_inicial` que reciba un `DataFrame` por parámetro y retorne un `DataFrame` indexado por nombre de columna con las siguientes columnas:

* `"tipo_dato"`
* `"faltantes"`
* `"porcentaje_faltantes"`

El porcentaje de faltantes debe estar en escala de 0 a 100.

La función debe retornar un `DataFrame`.

In [2]:
# Escribe tu respuesta en esta celda
import pandas as pd

def resumen_inicial(df):
    resumen = pd.DataFrame({
        "tipo_dato": df.dtypes,
        "faltantes": df.isnull().sum(),
        "porcentaje_faltantes": (df.isnull().sum() / len(df)) * 100
    })

    return resumen


In [ ]:
## AUTO-REVISION

mini = pd.DataFrame({
    "a": [1, np.nan, 3],
    "b": ["x", "y", None]
})

esperado_mini = pd.DataFrame({
    "tipo_dato": mini.dtypes.astype(str),
    "faltantes": mini.isna().sum(),
    "porcentaje_faltantes": mini.isna().mean() * 100
})

try:
    resumen_inicial
    assert type(resumen_inicial) == type(lambda: None)
except:
    raise NotImplementedError("No existe una función llamada resumen_inicial.")

try:
    respuesta = resumen_inicial(cargar_datos())
    respuesta_mini = resumen_inicial(mini)
except:
    raise RuntimeError("Tu función produce un error al ejecutarse.")

assert isinstance(respuesta, pd.DataFrame), "La función debe retornar un DataFrame."
assert list(respuesta.columns) == ["tipo_dato", "faltantes", "porcentaje_faltantes"], "Las columnas del resumen no son las esperadas."
assert list(respuesta.index) == list(cargar_datos().columns), "El índice del resumen debe corresponder a las columnas originales."
assert isinstance(respuesta_mini, pd.DataFrame), "Tu función debe funcionar también con otros DataFrame."
npt.assert_allclose(respuesta_mini["faltantes"].values, esperado_mini["faltantes"].values)
npt.assert_allclose(respuesta_mini["porcentaje_faltantes"].values, esperado_mini["porcentaje_faltantes"].values)
assert respuesta_mini["tipo_dato"].tolist() == esperado_mini["tipo_dato"].tolist(), "La columna tipo_dato es incorrecta."

print("Auto-revision superada.")

### Ejercicio 3

Implementa una función llamada `contar_duplicados_estudiante` que reciba un `DataFrame` por parámetro y retorne el número de registros duplicados según la columna `"estu_consecutivo"`.

La función debe retornar un entero.

In [3]:
# Escribe tu respuesta en esta celda
def contar_duplicados_estudiante(df):
    return df["estu_consecutivo"].duplicated().sum()


In [ ]:
## AUTO-REVISION

mini = pd.DataFrame({
    "estu_consecutivo": ["A", "B", "B", "C", "C", "C"],
    "x": [1, 2, 3, 4, 5, 6]
})

try:
    contar_duplicados_estudiante
    assert type(contar_duplicados_estudiante) == type(lambda: None)
except:
    raise NotImplementedError("No existe una función llamada contar_duplicados_estudiante.")

try:
    respuesta = contar_duplicados_estudiante(cargar_datos())
    respuesta_mini = contar_duplicados_estudiante(mini)
except:
    raise RuntimeError("Tu función produce un error al ejecutarse.")

assert isinstance(respuesta, (int, np.integer)), "La función debe retornar un número entero."
assert int(respuesta) == 379, "La cantidad de duplicados en el dataset base es incorrecta."
assert int(respuesta_mini) == 3, "La función no está contando bien duplicados en un caso de prueba simple."

print("Auto-revision superada.")

### Ejercicio 4

Implementa una función llamada `limpiar_datos` que reciba un `DataFrame` por parámetro y retorne una versión limpia siguiendo **exactamente** estas reglas:

**1.** Eliminar duplicados usando `"estu_consecutivo"` y conservar la primera aparición.  
**2.** Reemplazar faltantes en `"cole_bilingue"`, `"fami_tieneinternet"`, `"fami_tienecomputador"` y `"fami_estratovivienda"` por `"No reporta"`.  
**3.** Eliminar filas con faltantes en `"punt_global"`, `"punt_matematicas"` o `"punt_lectura_critica"`.  
**4.** Normalizar el texto en estas columnas usando la función `normalizar_texto`:  
`"cole_area_ubicacion"`, `"cole_naturaleza"`, `"cole_jornada"`, `"cole_depto_ubicacion"`, `"cole_mcpio_ubicacion"`, `"estu_genero"`, `"cole_calendario"`, `"cole_genero"`, `"desemp_ingles"`, `"fami_tienecomputador"`, `"fami_tieneinternet"`, `"fami_estratovivienda"` y `"cole_bilingue"`.

La función debe retornar un `DataFrame`.

In [5]:
# Escribe tu respuesta en esta celda
def limpiar_datos(df):
    # 1. Eliminar duplicados conservando la primera aparición
    df = df.drop_duplicates(subset="estu_consecutivo", keep="first")

    # 2. Reemplazar valores faltantes por "No reporta"
    columnas_no_reporta = [
        "cole_bilingue",
        "fami_tieneinternet",
        "fami_tienecomputador",
        "fami_estratovivienda"
    ]
    df[columnas_no_reporta] = df[columnas_no_reporta].fillna("No reporta")

    # 3. Eliminar filas con faltantes en las columnas de puntajes
    df = df.dropna(subset=[
        "punt_global",
        "punt_matematicas",
        "punt_lectura_critica"
    ])

    # 4. Normalizar texto
    columnas_texto = [
        "cole_area_ubicacion",
        "cole_naturaleza",
        "cole_jornada",
        "cole_depto_ubicacion",
        "cole_mcpio_ubicacion",
        "estu_genero",
        "cole_calendario",
        "cole_genero",
        "desemp_ingles",
        "fami_tienecomputador",
        "fami_tieneinternet",
        "fami_estratovivienda",
        "cole_bilingue"
    ]

    for columna in columnas_texto:
        df[columna] = df[columna].apply(normalizar_texto)

    return df

In [ ]:
## AUTO-REVISION

mini = pd.DataFrame({
    "periodo": [20201, 20201, 20202],
    "estu_consecutivo": ["A", "A", "B"],
    "cole_area_ubicacion": [" urbano ", " urbano ", "rural"],
    "cole_bilingue": [np.nan, np.nan, "n"],
    "cole_calendario": ["a", "a", "b"],
    "cole_genero": ["mixto", "mixto", "femenino"],
    "cole_jornada": ["mañana", "mañana", "tarde"],
    "cole_naturaleza": ["oficial", "oficial", "no oficial"],
    "cole_depto_ubicacion": ["Bogotá", "Bogotá", "Nariño"],
    "cole_mcpio_ubicacion": ["Bogotá D.C.", "Bogotá D.C.", "Pasto"],
    "estu_genero": ["f", "f", "m"],
    "fami_estratovivienda": [np.nan, np.nan, "Estrato 2"],
    "fami_personashogar": ["Cuatro", "Cuatro", "Tres"],
    "fami_tienecomputador": [np.nan, np.nan, "si"],
    "fami_tieneinternet": ["si", "si", np.nan],
    "desemp_ingles": ["a-", "a-", "b1"],
    "punt_ingles": [50, 50, 60],
    "punt_matematicas": [55, 55, np.nan],
    "punt_sociales_ciudadanas": [52, 52, 58],
    "punt_c_naturales": [53, 53, 59],
    "punt_lectura_critica": [57, 57, 61],
    "punt_global": [260, 260, 300],
})

try:
    limpiar_datos
    assert type(limpiar_datos) == type(lambda: None)
except:
    raise NotImplementedError("No existe una función llamada limpiar_datos.")

try:
    respuesta = limpiar_datos(cargar_datos())
    respuesta_mini = limpiar_datos(mini)
except:
    raise RuntimeError("Tu función produce un error al ejecutarse.")

assert isinstance(respuesta, pd.DataFrame), "La función debe retornar un DataFrame."
assert respuesta.shape == (1156, 22), "La base limpia no tiene la forma esperada."
assert respuesta.duplicated(subset=["estu_consecutivo"]).sum() == 0, "Aún hay duplicados según estu_consecutivo."
assert respuesta[["punt_global", "punt_matematicas", "punt_lectura_critica"]].isna().sum().sum() == 0, "No se eliminaron correctamente los faltantes críticos."
assert "BOGOTA" in set(respuesta["cole_depto_ubicacion"]), "La normalización de texto parece no haberse aplicado correctamente."
assert "NO REPORTA" in set(respuesta["cole_bilingue"]), "Debes completar faltantes con la etiqueta 'No reporta' y luego normalizar."

assert respuesta_mini.shape[0] == 1, "En el caso de prueba simple no se eliminaron correctamente duplicados o faltantes críticos."
fila = respuesta_mini.iloc[0]
assert fila["cole_area_ubicacion"] == "URBANO", "No normalizaste correctamente cole_area_ubicacion."
assert fila["cole_depto_ubicacion"] == "BOGOTA", "No normalizaste correctamente tildes o mayúsculas."
assert fila["cole_bilingue"] == "NO REPORTA", "No completaste correctamente faltantes en cole_bilingue."

print("Auto-revision superada.")

### Ejercicio 5

Implementa una función llamada `resumen_estadistico` que reciba el `DataFrame` limpio por parámetro y retorne un `Series` con estos índices exactos:

* `"media_punt_global"`
* `"mediana_punt_global"`
* `"moda_desemp_ingles"`
* `"desviacion_punt_global"`
* `"varianza_punt_global"`
* `"q1_punt_global"`
* `"q3_punt_global"`
* `"percentil_90_punt_global"`

Este ejercicio te permite conectar la estadística con preguntas como: ¿cuál es el desempeño típico?, ¿qué tan dispersos están los resultados?, ¿dónde empieza el 25% superior? y ¿qué valor marca el 10% más alto?

La función debe retornar un `Series`.

In [ ]:
# Escribe tu respuesta en esta celda
import pandas as pd

def resumen_estadistico(df):
    resumen = pd.Series({
        "media_punt_global": df["punt_global"].mean(),
        "mediana_punt_global": df["punt_global"].median(),
        "moda_desemp_ingles": df["desemp_ingles"].mode().iloc[0],
        "desviacion_punt_global": df["punt_global"].std(),
        "varianza_punt_global": df["punt_global"].var(),
        "q1_punt_global": df["punt_global"].quantile(0.25),
        "q3_punt_global": df["punt_global"].quantile(0.75),
        "percentil_90_punt_global": df["punt_global"].quantile(0.90)
    })

    return resumen

In [ ]:
## AUTO-REVISION

try:
    resumen_estadistico
    assert type(resumen_estadistico) == type(lambda: None)
except:
    raise NotImplementedError("No existe una función llamada resumen_estadistico.")

datos_limpios = limpiar_datos(cargar_datos())
esperado = pd.Series({
    "media_punt_global": datos_limpios["punt_global"].mean(),
    "mediana_punt_global": datos_limpios["punt_global"].median(),
    "moda_desemp_ingles": datos_limpios["desemp_ingles"].mode().iloc[0],
    "desviacion_punt_global": datos_limpios["punt_global"].std(),
    "varianza_punt_global": datos_limpios["punt_global"].var(),
    "q1_punt_global": datos_limpios["punt_global"].quantile(0.25),
    "q3_punt_global": datos_limpios["punt_global"].quantile(0.75),
    "percentil_90_punt_global": datos_limpios["punt_global"].quantile(0.90),
}, dtype="object")

try:
    respuesta = resumen_estadistico(datos_limpios)
except:
    raise RuntimeError("Tu función produce un error al ejecutarse.")

assert isinstance(respuesta, pd.Series), "La función debe retornar un Series."
assert list(respuesta.index) == list(esperado.index), "Los índices del Series no son los esperados."
assert respuesta["moda_desemp_ingles"] == esperado["moda_desemp_ingles"], "La moda de desempeño en inglés es incorrecta."
for indice in esperado.index:
    if indice != "moda_desemp_ingles":
        npt.assert_allclose(float(respuesta[indice]), float(esperado[indice]))

print("Auto-revision superada.")

### Ejercicio 6

Implementa una función llamada `resumen_area` que reciba el `DataFrame` limpio por parámetro y retorne un `DataFrame` agrupado por `"cole_area_ubicacion"` con estas columnas:

* `"promedio_punt_global"`
* `"mediana_punt_matematicas"`
* `"total_estudiantes"`

Ordena el resultado por índice de forma ascendente.

Este tipo de resumen ayuda a comparar grupos y a responder si existen diferencias entre contextos rurales y urbanos.

La función debe retornar un `DataFrame`.

In [ ]:
# Escribe tu respuesta en esta celda
import pandas as pd

def resumen_area(df):
    resumen = df.groupby("cole_area_ubicacion").agg(
        promedio_punt_global=("punt_global", "mean"),
        mediana_punt_matematicas=("punt_matematicas", "median"),
        total_estudiantes=("cole_area_ubicacion", "count")
    )

    return resumen.sort_index()


In [ ]:
## AUTO-REVISION

try:
    resumen_area
    assert type(resumen_area) == type(lambda: None)
except:
    raise NotImplementedError("No existe una función llamada resumen_area.")

datos_limpios = limpiar_datos(cargar_datos())
esperado = datos_limpios.groupby("cole_area_ubicacion").agg(
    promedio_punt_global=("punt_global", "mean"),
    mediana_punt_matematicas=("punt_matematicas", "median"),
    total_estudiantes=("estu_consecutivo", "count")
).sort_index()

try:
    respuesta = resumen_area(datos_limpios)
except:
    raise RuntimeError("Tu función produce un error al ejecutarse.")

assert isinstance(respuesta, pd.DataFrame), "La función debe retornar un DataFrame."
assert list(respuesta.columns) == list(esperado.columns), "Las columnas del resumen por área son incorrectas."
assert list(respuesta.index) == list(esperado.index), "El índice del resumen por área es incorrecto."
npt.assert_allclose(respuesta["promedio_punt_global"].values, esperado["promedio_punt_global"].values)
npt.assert_allclose(respuesta["mediana_punt_matematicas"].values, esperado["mediana_punt_matematicas"].values)
npt.assert_allclose(respuesta["total_estudiantes"].values, esperado["total_estudiantes"].values)

print("Auto-revision superada.")

### Ejercicio 7

Implementa una función llamada `top_departamentos` que reciba el `DataFrame` limpio por parámetro y un entero `n` opcional (por defecto `10`). La función debe retornar un `DataFrame` con los `n` departamentos con mayor `"promedio_punt_global"`, junto con la columna `"total_estudiantes"`.

Debes:

* agrupar por `"cole_depto_ubicacion"`;
* calcular `"promedio_punt_global"` y `"total_estudiantes"`;
* ordenar de mayor a menor por `"promedio_punt_global"` y luego por `"total_estudiantes"`;
* retornar solo los primeros `n`.

La función debe retornar un `DataFrame`.

In [ ]:
# Escribe tu respuesta en esta celda
import pandas as pd

def top_departamentos(df, n=10):
    resumen = df.groupby("cole_depto_ubicacion").agg(
        promedio_punt_global=("punt_global", "mean"),
        total_estudiantes=("cole_depto_ubicacion", "count")
    )

    resumen = resumen.sort_values(
        by=["promedio_punt_global", "total_estudiantes"],
        ascending=[False, False]
    )

    return resumen.head(n)


In [ ]:
## AUTO-REVISION

mini = pd.DataFrame({
    "cole_depto_ubicacion": ["A", "A", "B", "C", "C"],
    "punt_global": [200, 220, 250, 240, 260],
    "estu_consecutivo": ["x1", "x2", "x3", "x4", "x5"]
})

esperado_mini = mini.groupby("cole_depto_ubicacion").agg(
    promedio_punt_global=("punt_global", "mean"),
    total_estudiantes=("estu_consecutivo", "count")
).sort_values(["promedio_punt_global", "total_estudiantes"], ascending=[False, False]).head(2)

try:
    top_departamentos
    assert type(top_departamentos) == type(lambda: None)
except:
    raise NotImplementedError("No existe una función llamada top_departamentos.")

datos_limpios = limpiar_datos(cargar_datos())
esperado = datos_limpios.groupby("cole_depto_ubicacion").agg(
    promedio_punt_global=("punt_global", "mean"),
    total_estudiantes=("estu_consecutivo", "count")
).sort_values(["promedio_punt_global", "total_estudiantes"], ascending=[False, False]).head(10)

try:
    respuesta = top_departamentos(datos_limpios)
    respuesta_mini = top_departamentos(mini, n=2)
except:
    raise RuntimeError("Tu función produce un error al ejecutarse.")

assert isinstance(respuesta, pd.DataFrame), "La función debe retornar un DataFrame."
assert list(respuesta.columns) == list(esperado.columns), "Las columnas del top de departamentos son incorrectas."
assert len(respuesta) == 10, "La función debe retornar 10 departamentos por defecto."
assert respuesta.index[0] == esperado.index[0], "El departamento líder no coincide con el esperado."
npt.assert_allclose(respuesta["promedio_punt_global"].values, esperado["promedio_punt_global"].values)
npt.assert_allclose(respuesta["total_estudiantes"].values, esperado["total_estudiantes"].values)
assert list(respuesta_mini.index) == list(esperado_mini.index), "Tu función no ordena correctamente en el caso de prueba simple."

print("Auto-revision superada.")

### Ejercicio 8

Implementa una función llamada `resumen_normalidad` que reciba el `DataFrame` limpio por parámetro y retorne un `DataFrame` indexado por las variables `"punt_global"` y `"punt_ingles"` con estas columnas:

* `"media"`
* `"mediana"`
* `"desviacion_estandar"`
* `"asimetria"`
* `"aprox_normal"`

La columna `"aprox_normal"` debe ser booleana y valer `True` cuando `abs(asimetria) < 0.5`, y `False` en caso contrario.

Este ejercicio busca que razones sobre la normalidad: una variable más simétrica suele tener media y mediana más cercanas, mientras que una variable más sesgada exige una lectura más cuidadosa.

La función debe retornar un `DataFrame`.

In [ ]:
# Escribe tu respuesta en esta celda
import pandas as pd

def resumen_normalidad(df):
    variables = ["punt_global", "punt_ingles"]

    resumen = pd.DataFrame(index=variables)

    resumen["media"] = [df[var].mean() for var in variables]
    resumen["mediana"] = [df[var].median() for var in variables]
    resumen["desviacion_estandar"] = [df[var].std() for var in variables]
    resumen["asimetria"] = [df[var].skew() for var in variables]
    resumen["aprox_normal"] = resumen["asimetria"].abs() < 0.5

    return resumen

In [ ]:
## AUTO-REVISION

try:
    resumen_normalidad
    assert type(resumen_normalidad) == type(lambda: None)
except:
    raise NotImplementedError("No existe una función llamada resumen_normalidad.")

datos_limpios = limpiar_datos(cargar_datos())
variables = ["punt_global", "punt_ingles"]
esperado = pd.DataFrame({
    "media": datos_limpios[variables].mean(),
    "mediana": datos_limpios[variables].median(),
    "desviacion_estandar": datos_limpios[variables].std(),
    "asimetria": datos_limpios[variables].skew()
})
esperado["aprox_normal"] = esperado["asimetria"].abs() < 0.5

try:
    respuesta = resumen_normalidad(datos_limpios)
except:
    raise RuntimeError("Tu función produce un error al ejecutarse.")

assert isinstance(respuesta, pd.DataFrame), "La función debe retornar un DataFrame."
assert list(respuesta.index) == variables, "El índice del resumen de normalidad es incorrecto."
assert list(respuesta.columns) == list(esperado.columns), "Las columnas del resumen de normalidad son incorrectas."
for columna in ["media", "mediana", "desviacion_estandar", "asimetria"]:
    npt.assert_allclose(respuesta[columna].values, esperado[columna].values)
assert respuesta["aprox_normal"].tolist() == esperado["aprox_normal"].tolist(), "La interpretación booleana de normalidad no coincide."

print("Auto-revision superada.")

### Ejercicio 9

Implementa una función llamada `graficar_top_departamentos` que reciba el `DataFrame` retornado por `top_departamentos` y construya una **gráfica de barras horizontal** del `"promedio_punt_global"`.

La función debe:

* retornar el objeto `Axes`;
* usar como título exacto: `"Top 10 departamentos por puntaje global promedio"`;
* usar como eje `x`: `"Puntaje global promedio"`;
* usar como eje `y`: `"Departamento"`.

La función debe retornar un objeto `Axes`.

In [ ]:
# Escribe tu respuesta en esta celda
import matplotlib.pyplot as plt

def graficar_top_departamentos(df):
    ax = df["promedio_punt_global"].plot(
        kind="barh",
        figsize=(10, 6)
    )

    ax.set_title("Top 10 departamentos por puntaje global promedio")
    ax.set_xlabel("Puntaje global promedio")
    ax.set_ylabel("Departamento")

    return ax

NameError: name 'normalizar_texto' is not defined

In [ ]:
## AUTO-REVISION

try:
    graficar_top_departamentos
    assert type(graficar_top_departamentos) == type(lambda: None)
except:
    raise NotImplementedError("No existe una función llamada graficar_top_departamentos.")

tabla = top_departamentos(limpiar_datos(cargar_datos()))
plt.close("all")

try:
    ax = graficar_top_departamentos(tabla)
except:
    raise RuntimeError("Tu función produce un error al ejecutarse.")

assert isinstance(ax, Axes), "La función debe retornar un objeto Axes."
assert ax.get_title() == "Top 10 departamentos por puntaje global promedio", "El título del gráfico es incorrecto."
assert ax.get_xlabel() == "Puntaje global promedio", "La etiqueta del eje x es incorrecta."
assert ax.get_ylabel() == "Departamento", "La etiqueta del eje y es incorrecta."
assert len(ax.patches) == len(tabla), "La cantidad de barras no coincide con el tamaño de la tabla."

print("Auto-revision superada.")

### import matplotlib.pyplot as plt

def graficar_top_departamentos(df):
    ax = df["promedio_punt_global"].plot(
        kind="barh",
        figsize=(10, 6)
    )

    ax.set_title("Top 10 departamentos por puntaje global promedio")
    ax.set_xlabel("Puntaje global promedio")
    ax.set_ylabel("Departamento")

    return ax

In [11]:
# Escribe tu respuesta en esta celda
import matplotlib.pyplot as plt

def graficar_top_departamentos(df):
    # Crear la gráfica de barras horizontal
    ax = df["promedio_punt_global"].plot(
        kind="barh",
        figsize=(10, 6)
    )

    # Configurar título y etiquetas
    ax.set_title("Top 10 departamentos por puntaje global promedio")
    ax.set_xlabel("Puntaje global promedio")
    ax.set_ylabel("Departamento")

    # Retornar el objeto Axes
    return ax

ModuleNotFoundError: No module named 'matplotlib'

In [ ]:
## AUTO-REVISION

try:
    tabla_calor_jornada_naturaleza
    assert type(tabla_calor_jornada_naturaleza) == type(lambda: None)
except:
    raise NotImplementedError("No existe una función llamada tabla_calor_jornada_naturaleza.")

datos_limpios = limpiar_datos(cargar_datos())
esperado = datos_limpios.pivot_table(
    index="cole_jornada",
    columns="cole_naturaleza",
    values="punt_global",
    aggfunc="mean"
).sort_index()

try:
    respuesta = tabla_calor_jornada_naturaleza(datos_limpios)
except:
    raise RuntimeError("Tu función produce un error al ejecutarse.")

assert isinstance(respuesta, pd.DataFrame), "La función debe retornar un DataFrame."
respuesta = respuesta.sort_index()
assert list(respuesta.index) == list(esperado.index), "El índice de la tabla de calor es incorrecto."
assert list(respuesta.columns) == list(esperado.columns), "Las columnas de la tabla de calor son incorrectas."
npt.assert_allclose(respuesta.values, esperado.values)

print("Auto-revision superada.")

### Ejercicio 11

Implementa una función llamada `graficar_heatmap` que reciba la tabla retornada por `tabla_calor_jornada_naturaleza` y construya una visualización tipo **heatmap** o mapa de calor.

La función debe:

* retornar un objeto `Axes`;
* usar como título exacto: `"Puntaje global promedio por jornada y naturaleza"`;
* usar como eje `x`: `"Naturaleza del colegio"`;
* usar como eje `y`: `"Jornada"`.

Puedes usar `imshow`, `matshow` o cualquier otra estrategia con `matplotlib`.

In [16]:
# Escribe tu respuesta en esta celda
try:
    import matplotlib.pyplot as plt
except ModuleNotFoundError:
    print("Error: La librería 'matplotlib' no está instalada.")
    raise

def graficar_heatmap(tabla):
    fig, ax = plt.subplots(figsize=(8, 6))

    imagen = ax.imshow(tabla.values, cmap="viridis", aspect="auto")

    ax.set_xticks(range(len(tabla.columns)))
    ax.set_xticklabels(tabla.columns, rotation=45, ha="right")

    ax.set_yticks(range(len(tabla.index)))
    ax.set_yticklabels(tabla.index)

    ax.set_title("Puntaje global promedio por jornada y naturaleza")
    ax.set_xlabel("Naturaleza del colegio")
    ax.set_ylabel("Jornada")

    plt.colorbar(imagen, ax=ax)
    plt.tight_layout()

    return ax

Error: La librería 'matplotlib' no está instalada.


ModuleNotFoundError: No module named 'matplotlib'

In [ ]:
## AUTO-REVISION

try:
    graficar_heatmap
    assert type(graficar_heatmap) == type(lambda: None)
except:
    raise NotImplementedError("No existe una función llamada graficar_heatmap.")

tabla = tabla_calor_jornada_naturaleza(limpiar_datos(cargar_datos()))
plt.close("all")

try:
    ax = graficar_heatmap(tabla)
except:
    raise RuntimeError("Tu función produce un error al ejecutarse.")

assert isinstance(ax, Axes), "La función debe retornar un objeto Axes."
assert ax.get_title() == "Puntaje global promedio por jornada y naturaleza", "El título del heatmap es incorrecto."
assert ax.get_xlabel() == "Naturaleza del colegio", "La etiqueta del eje x es incorrecta."
assert ax.get_ylabel() == "Jornada", "La etiqueta del eje y es incorrecta."

print("Auto-revision superada.")

## Retos creativos sugeridos

Si ya completaste el taller, puedes profundizar con estas visualizaciones adicionales:

**1.** Construye un **boxplot** de `punt_global` por `cole_area_ubicacion` para comparar dispersión entre lo rural y lo urbano.  
**2.** Crea una **dispersión** entre `punt_matematicas` y `punt_lectura_critica`, usando colores distintos según `fami_tieneinternet`.  
**3.** Diseña una gráfica que compare el `punt_global` promedio por `fami_estratovivienda`. Si lo deseas, ordénala de menor a mayor para facilitar la lectura.  
**4.** Construye un histograma de `punt_ingles` y compáralo con el de `punt_global`. Luego responde: ¿cuál de las dos variables parece más cercana a una distribución normal?

## Cierre pedagógico

Si llegaste hasta aquí, ya aplicaste el flujo completo de la semana 4:

* leíste una base real;
* exploraste su estructura;
* detectaste faltantes y duplicados;
* limpiaste y preparaste la información;
* calculaste estadísticas descriptivas;
* razonaste sobre normalidad y asimetría;
* construiste visualizaciones orientadas a la toma de decisiones.

Ese es exactamente el tipo de proceso que precede a análisis más complejos y a modelos posteriores.